# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Research question:** Which pages that show up in search get fewer clicks than similar pages ranked at a similar position, and are worth a content reviewer's time to check first?

**The decision this supports:** Content reviewers have limited time and can't manually check every page. This work helps them decide which pages to look at first when they don't have time to check everything.

**Who acts on it:** Content reviewers, who can look at flagged pages and decide whether to change the title or meta description.

**Cost of a wrong call:** If I flag a page that's actually fine, the reviewer loses a few minutes checking it low cost. If I miss a real problem page, it just keeps underperforming, a missed opportunity, not a disaster. Neither error is severe, which is part of why this is a reasonable place to apply a scoring approach rather than something higher-stakes.

**Why data/ML helps here at all:** A low click count can be completely normal or genuinely bad, depending on the page's position in search results. Comparing that fairly across thousands of pages, accounting for position, is something a computer can do consistently and quickly, Doing it by eye across a large dataset isn't realistic.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Source and scope:** I used FlyRank's Hugging Face-hosted warehouse, scoped to month=2026-03 (a safe middle month), deliberately not the most recent month, since that's reserved as a sealed test set I shouldn't touch while developing my own logic.

**Grain, verified with real queries:** One row = one page's performance on one day. March 2026 has 9,841,378 rows spanning March 1-31, across 331,437 distinct pages. I confirmed the grain with a group-by check (page + client + date) that returned zero duplicate groups.

**A real limit in this data:** Only 3,611,061 rows (36.7%) actually have usable search data (gsc_data_available = TRUE). The rest can't support search-position analysis. This means pages/clients with short or missing search history are effectively invisible in my slice. My findings may not apply evenly across all of FlyRank's clients.

**The working slice used throughout my baseline, model, and validation work:** Pages with impressions_90d >= 500 and avg_position between 1-20 are "actually visible in search", which narrows to 12,023 pages.

**What I deliberately excluded, and why:**
- `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`,`ai_other`, AI-assistant referral columns, not relevant to my CTR/position question.
- `trend_direction`, `trend_pct`, flagged in the data dictionary as label-source fields, never safe as features.
- `ctr`, `clicks_90d`, `clicks_last_30d`, `clicks_prev_30d`, these define my 
  own label (needs_review), so using them as features would be circular
- Post-click behavior columns (`pageviews_90d`, `sessions_90d`, `engagement_rate`, `scroll_rate`, etc.), these only exist after someone already clicked, so they're downstream of the outcome I'm trying to predict,not a legitimate input.
- FlyRank's own product flags (`health_score`, `priority_score`, `needs_ctr_fix`, `is_quick_win`), confirmed these don't even exist in this dataset.

**Public-safe:** all IDs (`content_id`, `client_id`) are pseudonymous hashes, used only for grouping my train/test split, never as model features, and never printed alongside anything identifying.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Baseline (ML-07):** I first built a simple rule before touching any model. The score is: how far a page's CTR falls below the median CTR of other pages in its own position tier (tier_median_ctr - page_ctr, floored at 0).

**Reason code**: 
ctr_below_tier_peers.
**Action**:
review_title_and_meta_description.

**Two signal checks shaped this rule, not the other way around.** Signal A tested 
whether CTR actually drops as position gets worse — verdict MIXED (striking was 
clearly worse, but top_3 didn't beat page_1 as expected, likely due to a small 
sample of 458 pages). Signal B tested whether high-traffic pages have more room 
to improve — verdict OPPOSITE (high-traffic pages were already outperforming 
their tier, not underperforming). Because of Signal B, I never multiply my score 
by traffic volume — I only use impressions_90d >= 500 as an eligibility gate.

**Model (ML-08):** Logistic Regression, chosen because my question is "which pages 
to check first" (a ranking question), not a fixed label — a classifier's 
predicted probability gives a natural ranking, and Logistic Regression's 
coefficients are readable enough to explain in plain words.

**Label:** since no reviewer has manually confirmed which pages truly have a 
title problem, I used a proxy label — the top 25% of CTR gaps (needs_review = 1). 
This is grounded in a real, observed number, not someone else's rule.

**Features:** 20 columns describing the page before anyone clicks (search_volume, 
competition, content_type, avg_position, position_tier, freshness_tier, etc.). 
I deliberately excluded ctr, all click-based columns, post-click behavior columns, 
and trend_direction/trend_pct — using any of these would let the model just read 
back its own label instead of learning a real pattern.

**Split:** grouped by client_id, not random. I confirmed zero client overlap 
between train and test — this matters because a random split let the same client's 
pages leak between train and test, which I later proved inflates the score 
artificially (see Results).

**Leakage checks, run twice:** first in ML-04, where I deliberately added a 
"cheat" feature built from the answer itself — my honest model scored 0.0158, 
but adding the cheat column jumped it to a suspicious 1.0000, confirming the 
leak trap works. Second, in ML-09, I re-checked my final 20 ML-08 features 
against three leakage categories (label-derived, future/overlapping-window, 
product-flag columns) — all three checks came back empty, confirming the 
features actually used in my model are clean.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.